<a href="https://colab.research.google.com/github/NehaBongarde2004/Deep-Learning/blob/main/DL_Exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

In [2]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize data (0–255 → 0–1)
x_train = x_train / 255.0
x_test = x_test / 255.0

# Convert labels to categorical
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print("Training shape:", x_train.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Training shape: (50000, 32, 32, 3)


In [3]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
for layer in base_model.layers:
    layer.trainable = False

In [5]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

In [6]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_frozen = model.fit(
    x_train, y_train,
    epochs=5,
    validation_data=(x_test, y_test)
)

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 258s 159ms/step - accuracy: 0.1010 - loss: 2.3087 - val_accuracy: 0.1044 - val_loss: 2.2994
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 241s 154ms/step - accuracy: 0.1133 - loss: 2.2888 - val_accuracy: 0.1676 - val_loss: 2.2595
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.1318 - loss: 2.2632

In [ ]:
loss_frozen, acc_frozen = model.evaluate(x_test, y_test)
print("Frozen Model Accuracy:", acc_frozen)

In [ ]:
for layer in base_model.layers[-10:]:
    layer.trainable = True

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
loss_ft, acc_ft = model.evaluate(x_test, y_test)
print("Fine-Tuned Model Accuracy:", acc_ft)

313/313 ━━━━━━━━━━━━━━━━━━━━ 51s 148ms/step - accuracy: 0.1000 - loss: 2.3026
Fine-Tuned Model Accuracy: 0.10000000149011612


In [ ]:
history_finetune = model.fit(
    x_train, y_train,
    epochs=5, # You can adjust the number of epochs for fine-tuning
    validation_data=(x_test, y_test)
)

plt.plot(history_frozen.history['val_accuracy'], label='Frozen')
plt.plot(history_finetune.history['val_accuracy'], label='Fine-tuned')
plt.title('Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.show()

Epoch 1/5
   7/1563 ━━━━━━━━━━━━━━━━━━━━ 10:46 415ms/step - accuracy: 0.1143 - loss: 2.3349